# Feature Engineering Notebook

This notebook creates derived features for customer retention analysis.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('c:\\Users\\LENOVO\\Downloads\\customers_cleaned.csv')
print(df.shape)

(3900, 18)


In [3]:
df['promo_dependency_score'] = (
    df['discount_applied'] + df['promo_code_used']
) / 2

print(df['promo_dependency_score'].value_counts())

promo_dependency_score
0.0    2223
1.0    1677
Name: count, dtype: int64


In [4]:
df['value_score'] = (
    df['purchase_amount_usd'] * 0.6 +
    df['previous_purchases'] * 2 * 0.4
)

df['value_tier'] = pd.qcut(
    df['value_score'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Premium']
)

print(df['value_tier'].value_counts())

value_tier
Medium     979
Low        976
High       974
Premium    971
Name: count, dtype: int64


In [5]:
df['satisfaction_flag'] = (df['review_rating'] >= 4).astype(int)
print("Satisfied customers:", df['satisfaction_flag'].sum())

Satisfied customers: 1634


In [6]:
frequency_map = {
    'Weekly': 52,
    'Fortnightly': 26,
    'Bi-Weekly': 26,
    'Monthly': 12,
    'Quarterly': 4,
    'Every 3 Months': 4,
    'Annually': 1
}

df['frequency_score'] = df['frequency_of_purchases'].map(frequency_map)
print(df['frequency_score'].value_counts())

frequency_score
4     1147
26    1089
1      572
12     553
52     539
Name: count, dtype: int64


In [9]:
df.to_csv('../data/cleaned/customers_features.csv', index=False)
print("Done! Columns now:")
print(df.columns.tolist())

Done! Columns now:
['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_usd', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases', 'promo_dependency_score', 'value_score', 'value_tier', 'satisfaction_flag', 'frequency_score', 'loyalty_def_A', 'loyalty_def_B', 'segment']


In [8]:
def assign_segment(row):
    if row['promo_dependency_score'] == 1.0:
        return 'Promo-Driven'
    elif row['frequency_score'] >= 26 and row['promo_dependency_score'] == 0:
        return 'Loyal'
    elif row['previous_purchases'] <= 2:
        return 'One-Time'
    else:
        return 'Casual'

df['segment'] = df.apply(assign_segment, axis=1)
print(df['segment'].value_counts())

segment
Promo-Driven    1677
Casual          1242
Loyal            925
One-Time          56
Name: count, dtype: int64


In [7]:
df['loyalty_def_A'] = (
    (df['frequency_score'] >= 12) &
    (df['promo_dependency_score'] < 0.5)
).astype(int)

df['loyalty_def_B'] = (
    (df['previous_purchases'] > df['previous_purchases'].median()) &
    (df['satisfaction_flag'] == 1)
).astype(int)

print("Loyalty A vs Spend:", 
      df['loyalty_def_A'].corr(df['purchase_amount_usd']).round(3))
print("Loyalty B vs Spend:", 
      df['loyalty_def_B'].corr(df['purchase_amount_usd']).round(3))

Loyalty A vs Spend: 0.001
Loyalty B vs Spend: 0.033
